# Классификация SI > медианы

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import RandomizedSearchCV

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report

In [2]:
# Загрузка данных
df = pd.read_excel('../data/Данные_для_курсовои_Классическое_МО.xlsx')
df = df.drop(columns=['Unnamed: 0'])
df = df.dropna()

In [3]:
# Подготовка данных
median_IC50 = df['SI'].median()                          # Медианные значения для бинаризации
df['target'] = (df['SI'] > median_IC50).astype(int)      # Целевая переменные

print(f"Баланс классов: {df['target'].value_counts(normalize=True)[1]*100:.1f}%")

X = df.drop(columns=['IC50, mM', 'CC50, mM', 'SI', 'target'])
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

Баланс классов: 50.0%


Баланс классов выглядит очень хорошо

In [4]:
# Удаление коррелированных признаков
corr = X_train.corr()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
to_drop = [col for col in upper.columns if any(abs(upper[col]) > 0.95)]
X_train = X_train.drop(columns=to_drop)
X_test = X_test.drop(columns=to_drop)

print(f"Признаков после отбора: {X_train.shape[1]}")

Признаков после отбора: 178


Применим следующие модели:  
- "LinearRegression"
- "RandomForest"
- "CatBoost"
- "XGBoost"

**Подбор гиперпараметров не проводился в полном объёме ввиду высокой вычислительной сложности**

In [5]:
# Модели классификации (без подбора гиперпараметров )
clf_models = {
    "Logistic Regression": Pipeline([
        ('scaler', RobustScaler()),
        ('model', LogisticRegression(
            random_state=42,
            max_iter=5000,
            solver='liblinear',
            class_weight='balanced'
        ))
    ]),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        random_state=42,
        n_jobs=-1,
        class_weight='balanced'
    ),
    "CatBoost": CatBoostClassifier(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        verbose=0,
        random_seed=42,
        auto_class_weights='Balanced'
    ),
    "XGBoost": XGBClassifier(  # Добавим
        n_estimators=200,
        max_depth=5,
        learning_rate=0.05,
        random_state=42,
        n_jobs=-1,
        eval_metric='logloss'
    )
}

# Обучение БЕЗ подбора гиперпараметров
results_si_clf = []

for name, model in clf_models.items():    
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    results_si_clf.append({
        'model': name,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
        'roc_auc': roc_auc_score(y_test, y_pred_proba)
    })

results_si_clf_df = pd.DataFrame(results_si_clf).sort_values('roc_auc', ascending=False)

print("\n" + "="*67)
print("РЕЗУЛЬТАТЫ КЛАССИФИКАЦИИ SI > МЕДИАНЫ:")
print("="*67)
print(results_si_clf_df[['model', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc']].to_string(index=False))
print(f"\n🏆 Лучшая модель: {results_si_clf_df.iloc[0]['model']} (ROC-AUC = {results_si_clf_df.iloc[0]['roc_auc']:.4f})")


РЕЗУЛЬТАТЫ КЛАССИФИКАЦИИ SI > МЕДИАНЫ:
              model  accuracy  precision  recall       f1  roc_auc
      Random Forest      0.62   0.633333    0.57 0.600000  0.68215
            XGBoost      0.63   0.627451    0.64 0.633663  0.66955
           CatBoost      0.62   0.627660    0.59 0.608247  0.65835
Logistic Regression      0.58   0.590909    0.52 0.553191  0.61445

🏆 Лучшая модель: Random Forest (ROC-AUC = 0.6822)


**Итог**

Построена модель бинарной классификации для определения, превышает ли значение SI медиану выборки. Лучшая модель — Random Forest (ROC-AUC ≈ 0.682), однако качество классификации умеренное, что указывает на более сложную зависимость и худшую разделимость классов